# Hate-speech context experiments: guided runner

This is the starting point for the project. It runs the research experiments from the reusable `src/` code without requiring you to understand the codebase first.

**First successful run:** attach this repository in Kaggle, leave the defaults unchanged, then choose **Run All**. The included Base and Context CSVs are used automatically. Google Cloud credentials are not needed for that first run.

> Content warning: the datasets and generated context contain hateful, racist, sexist, and otherwise offensive material for research purposes.

## What each choice means

- **Dataset**: `latent` evaluates tweets; `mami` evaluates textualised memes.
- **Experiment**: `zero-context` is the recommended first baseline. `rel` uses REL/Wikipedia and requires Kaggle internet access. `conceptnet` needs Numberbatch vectors. `append-embed`, `embed-concat`, and `context-embed` use generated context. `llm-enhance` and `direct-llm` require outputs from a completed Vertex batch job.
- **Task**: use `binary` first. Latent Hatred also supports `multiclass` subtype evaluation and MAMI supports `multilabel`.

- **Context source**: `included` uses the checked-in CSV in `data/`. `rerun` submits a fresh Vertex batch and later writes a new CSV in `artifacts/results/`; it never overwrites the included research artefacts.

After the first baseline, change only `experiment` to `embed-concat` to use the included LLM-generated context. For MAMI, change `dataset` to `mami` and provide the MAMI CSV path.

The notebook stops before model loading if a required file is missing and tells you the exact path to configure.

In [ ]:
from pathlib import Path
import os
import sys

# Find the repository whether this notebook is run locally or from /kaggle/input.
candidate_roots = [Path.cwd()]
kaggle_input = Path('/kaggle/input')
if kaggle_input.is_dir():
    candidate_roots.extend(kaggle_input.iterdir())
REPO_ROOT = next((path for path in candidate_roots if (path / 'src').is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError('Attach this repository as a Kaggle dataset, then run this cell again.')

# Kaggle input datasets are read-only; model downloads must go under /kaggle/working.
cache_root = Path('/kaggle/working/contextual-hsd-cache') if kaggle_input.is_dir() else REPO_ROOT / '.cache' / 'huggingface'
cache_root.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(cache_root)

if not kaggle_input.is_dir() and sys.version_info[:2] > (3, 12):
    raise RuntimeError(
        'Python 3.14 cannot install this project\'s Kaggle-pinned NumPy and PyTorch wheels. '
        'Use Kaggle, or restart with Python 3.10–3.12 (Python 3.10 is the documented runtime).'
    )

# uv-created environments can omit pip; make the notebook self-sufficient.
try:
    import pip
except ImportError:
    import ensurepip
    ensurepip.bootstrap(upgrade=True)

%pip install -q -r {REPO_ROOT / 'requirements.txt'}
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f'Using repository: {REPO_ROOT}')

In [2]:
from src import *
import os
import torch

# -------- Edit only this block for a normal run --------
RUN = {
    'dataset': 'latent',          # 'latent' or 'mami'
    'experiment': 'zero-context', # see EXPERIMENTS above
    'task': 'binary',             # Latent: 'binary'/'multiclass'; MAMI: 'binary'/'multilabel'
    'device': 'auto',             # 'auto' uses CUDA when available, otherwise CPU
    # The included Base and Context CSVs are the defaults. Set paths only to override them.
    'latent_hatred_dir': None,    # optional raw-stage fallback, e.g. '/kaggle/input/implicit-hate-corpus'
    'mami_csv': None,             # optional MAMI Base CSV override
    'conceptnet_path': None,      # only for 'conceptnet'
    'context_source': 'included', # 'included' (default) or 'rerun'
    'context_csv': None,          # generated CSV to use after a Vertex collection step
    'vertex_action': 'off',       # 'off', 'submit-context', or 'collect-context'
    'vertex_manifest_path': None, # set after submit-context, for collect-context
    # Non-secret cloud settings, needed only when vertex_action is not 'off'.
    'gcp_project': None,          # e.g. 'my-google-cloud-project'
    'gcs_bucket': None,           # e.g. 'my-contextual-hsd-bucket'
}
# ------------------------------------------------------

if RUN['device'] == 'auto':
    os.environ['HSD_DEVICE'] = 'cuda' if torch.cuda.is_available() else 'cpu'
elif RUN['device'] in {'cpu', 'cuda'}:
    os.environ['HSD_DEVICE'] = RUN['device']
else:
    raise ValueError("device must be 'auto', 'cpu', or 'cuda'")

overrides = {
    'HSD_LATENT_HATRED_DIR': RUN['latent_hatred_dir'],
    'HSD_MAMI_CSV': RUN['mami_csv'],
    'HSD_CONCEPTNET_PATH': RUN['conceptnet_path'],
    'HSD_LATENT_CONTEXT_CSV': RUN['context_csv'],
    'HSD_MAMI_CONTEXT_CSV': RUN['context_csv'],
    'HSD_ARTIFACT_DIR': '/kaggle/working/contextual-hsd-artifacts' if kaggle_input.is_dir() else str(REPO_ROOT / 'artifacts'),
    'HSD_GCP_PROJECT': RUN['gcp_project'],
    'HSD_GCS_BUCKET': RUN['gcs_bucket'],
}
for name, value in overrides.items():
    if value:
        os.environ[name] = str(value)
if RUN['context_source'] == 'included':
    os.environ.pop('HSD_LATENT_CONTEXT_CSV', None)
    os.environ.pop('HSD_MAMI_CONTEXT_CSV', None)

config = ExperimentConfig.from_environment(REPO_ROOT)
config.paths.create_artifact_directories()
print(f'Using device: {config.training.device}')
RUN

Using device: cpu


{'dataset': 'latent',
 'experiment': 'zero-context',
 'task': 'binary',
 'device': 'auto',
 'latent_hatred_dir': None,
 'mami_csv': None,
 'conceptnet_path': None,
 'context_source': 'included',
 'context_csv': None,
 'vertex_action': 'off',
 'vertex_manifest_path': None,
 'gcp_project': None,
 'gcs_bucket': None}

In [3]:
DATASET = RUN['dataset']
EXPERIMENT = RUN['experiment']
TASK = RUN['task']

if RUN['context_source'] not in {'included', 'rerun'}:
    raise ValueError("context_source must be 'included' or 'rerun'")
if RUN['vertex_action'] not in {'off', 'submit-context', 'collect-context'}:
    raise ValueError("vertex_action must be 'off', 'submit-context', or 'collect-context'")

# Fresh context does not need a prior context CSV at submission time.
preflight_experiment = 'zero-context' if RUN['context_source'] == 'rerun' and not RUN['context_csv'] else EXPERIMENT
validate_run_inputs(config.paths, DATASET, preflight_experiment)
if DATASET == 'latent':
    data = load_latent_hatred(config.paths)
    split = create_splits(data, 'class', config.training)
    if TASK == 'binary':
        LABELS = 'binary_class'
    elif TASK == 'multiclass':
        LABELS = 'implicit_class'
    else:
        raise ValueError("Latent Hatred task must be 'binary' or 'multiclass'")
    MULTILABEL, CONTEXT_PATH = False, config.paths.latent_context_csv
elif DATASET == 'mami':
    data = load_mami(config.paths)
    split = create_splits(data, 'misogyny', config.training)
    if TASK == 'binary':
        LABELS, MULTILABEL = 'misogyny', False
    elif TASK == 'multilabel':
        LABELS, MULTILABEL = MAMI_LABEL_COLUMNS, True
    else:
        raise ValueError("MAMI task must be 'binary' or 'multilabel'")
    CONTEXT_PATH = config.paths.mami_context_csv
else:
    raise ValueError("dataset must be 'latent' or 'mami'")

if RUN['context_source'] == 'rerun' and RUN['context_csv']:
    CONTEXT_PATH = Path(RUN['context_csv'])

print(f'Loaded {DATASET}: {len(split.train)} training rows and {len(split.test)} test rows')

Loaded latent: 17184 training rows and 4296 test rows


In [4]:
needs_context = EXPERIMENT in {'append-embed', 'embed-concat', 'context-embed'}
if needs_context and RUN['context_source'] == 'rerun' and not RUN['context_csv']:
    print("Fresh context has not been collected yet. Set vertex_action to 'submit-context', run the Vertex cell, wait for completion, then set it to 'collect-context'.")
elif EXPERIMENT == 'direct-llm':
    raise ValueError("direct-llm needs predictions from a completed Vertex batch.")
elif EXPERIMENT == 'llm-enhance':
    raise ValueError("llm-enhance needs enhanced text from a completed Vertex batch.")
else:
    sentence_model = load_sentence_model(config.model, device=config.training.device)
    embedder = lambda values: embed_texts(sentence_model, values, device=config.training.device)
    run_kwargs = {'embedder': embedder}

    if EXPERIMENT == 'rel':
        run_kwargs['context_provider'] = RelWikipediaContextProvider.create_default().extract
    elif needs_context:
        if DATASET == 'mami' and RUN['context_source'] == 'included':
            context = data[['meme id', 'post']].merge(
                load_context_csv(CONTEXT_PATH, 'meme id')[['meme id', 'response']], on='meme id', how='left'
            )
            context['response'] = context['response'].fillna('')
        else:
            context = load_context_csv(CONTEXT_PATH, 'post')
        context_by_post = context.drop_duplicates('post').set_index('post')['response'].to_dict()
        run_kwargs['context_provider'] = lambda post: context_by_post.get(post, '')
        if EXPERIMENT == 'context-embed':
            run_kwargs['context_embedder'] = load_context_embedder(config.model, config.training.device)
    elif EXPERIMENT == 'conceptnet':
        numberbatch = load_conceptnet_embeddings(config.paths.conceptnet_path)
        run_kwargs['vector_provider'] = lambda texts: conceptnet_context_vectors(texts, numberbatch)

    result = run_experiment(EXPERIMENT, split.train, split.test, LABELS, config.training, multilabel=MULTILABEL, **run_kwargs)
    print(format_evaluation(result))

c:\Users\joshw\Personal\contextual-hsd\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

## Reading the result

The final line is the macro-average precision, recall, and F1 score across the task labels. Compare runs only when dataset, task, split seed, and experiment name are the same. These are research-benchmark results, not moderation-policy recommendations.

For a first comparison, run `zero-context` and then `embed-concat` with the same dataset and task. The included context CSVs make that second run possible without making a new LLM request.

## Re-run LLM context (optional)

The default is to use the included context data. To create a fresh version, set context_source to rerun, fill in gcp_project and gcs_bucket, and add a Kaggle secret named GOOGLE_API_KEY. Set vertex_action to submit-context and run this cell. The output gives a manifest path. After the job succeeds, paste that path into vertex_manifest_path, set vertex_action to collect-context, and run this cell again. It writes a new CSV under artifacts/results; paste that path into context_csv, leave context_source as rerun, set vertex_action to off, and re-run from the configuration cell. For MAMI image requests, upload the source images to gs://<gcs_bucket>/MAMI_images/ first.

In [ ]:
if RUN['vertex_action'] != 'off':
    from kaggle_secrets import UserSecretsClient
    from google import genai
    from google.cloud import storage

    if not RUN['gcp_project'] or not RUN['gcs_bucket']:
        raise ValueError("Set gcp_project and gcs_bucket in RUN before using Vertex.")

    secrets = UserSecretsClient()
    _ = secrets.get_secret('GOOGLE_API_KEY')
    secrets.set_gcloud_credentials(project=config.vertex.project_id)
    storage_client = storage.Client(config.vertex.project_id)
    genai_client = genai.Client(vertexai=True, project=config.vertex.project_id, location=config.vertex.location)

    if DATASET == 'latent':
        system = 'You provide brief, factual background context for social-media posts. Identify implicit meaning, sarcasm, obscure entities, and relevant world knowledge without assigning a hate-speech label.\n\n'
        request = 'Give background context about the following tweet:\n\n'
        requests = build_text_requests(data['post'], system, request)
        result_kind = 'text'
    else:
        system = 'You provide brief, factual background context for memes. Identify implicit meaning, sarcasm, obscure entities, and relevant world knowledge without assigning a misogyny label.\n\n'
        request = 'Give background context about the following meme:\n\n'
        requests = build_image_requests(data['meme id'], config.vertex, system, request)
        result_kind = 'image'

    if RUN['vertex_action'] == 'submit-context':
        manifest = submit_requests('context', requests, config.paths, config.vertex, storage_client, genai_client)
        print(f'Batch submitted: {manifest.job_name}')
        print(f'When it succeeds, set vertex_manifest_path to: {manifest.local_manifest_path}')
    else:  # collect-context
        if not RUN['vertex_manifest_path']:
            raise ValueError("Set vertex_manifest_path to the manifest printed by submit-context.")
        manifest = load_manifest(Path(RUN['vertex_manifest_path']))
        state = batch_status(manifest, genai_client)
        print(f'Batch state: {state}')
        if 'SUCCEEDED' in state:
            manifest, context = collect_result(manifest, config.paths, config.vertex, storage_client, result_kind)
            if DATASET == 'latent':
                context['post'] = context['post'].str.removeprefix(system + request)
            else:
                context = data[['meme id', 'post']].merge(context, on='meme id', how='inner')[['post', 'response']]
            context_path = config.paths.artifact_dir / 'results' / f'{manifest.run_id}-context.csv'
            context.to_csv(context_path, index=False)
            print(f'Context CSV written: {context_path}')
            print("Set context_csv to this path, set vertex_action to 'off', and re-run from the configuration cell.")
else:
    print('Using included context data (or no context). This is expected for the recommended first run.')